# ML1: Predict the price of the Uber ride from a given pickup point to the
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;agreed drop-off location. Perform following tasks:  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;1. Pre-process the dataset.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;2. Identify outliers.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;3. Check the correlation.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;4. Implement linear regression and random forest regression models.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5. Evaluate the models and compare their respective scores like R2,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;RMSE, etc.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Dataset: https://www.kaggle.com/datasets/yasserh/uber-fares-dataset  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(download "uber.csv" and keep it in the same folder as this file)  

## Theory

### 1. MACHINE LEARNING (Unit I)
&nbsp;&nbsp;&nbsp;"A computer program is said to learn from experience E with respect to  
&nbsp;&nbsp;&nbsp;some task T and performance measure P, if its performance at T, as measured  
&nbsp;&nbsp;&nbsp;by P, improves with experience E." - Tom Mitchell.  
&nbsp;&nbsp;&nbsp;Traditional programming: Data + Program -&gt; Output  
&nbsp;&nbsp;&nbsp;Machine learning       : Data + Output  -&gt; Program (model)  
&nbsp;&nbsp;&nbsp;AI (broad goal of intelligent machines) &gt; ML (learning from data) &gt;  
&nbsp;&nbsp;&nbsp;Deep Learning (multi-layer neural nets). Data Science = extracting insight  
&nbsp;&nbsp;&nbsp;from data using statistics + ML + domain knowledge.  
&nbsp;&nbsp;&nbsp;Types of learning: SUPERVISED (labelled data: regression, classification),  
&nbsp;&nbsp;&nbsp;UNSUPERVISED (unlabelled: clustering), SEMI-SUPERVISED, REINFORCEMENT  
&nbsp;&nbsp;&nbsp;(agent learns via rewards).  
&nbsp;&nbsp;&nbsp;This problem is SUPERVISED REGRESSION (target fare\_amount is continuous).  

### 2. DATA PRE-PROCESSING (Unit II - Feature Engineering)
&nbsp;&nbsp;&nbsp;- Handling missing values: drop rows, or impute with mean/median/mode.  
&nbsp;&nbsp;&nbsp;- Removing invalid data: fare &lt;= 0, coordinates outside valid ranges,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;passenger\_count = 0 or &gt; 6.  
&nbsp;&nbsp;&nbsp;- Feature extraction: from pickup\_datetime derive hour, day, month, year,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;weekday; from coordinates derive trip DISTANCE using the HAVERSINE  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;formula:  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a = sin^2(dlat/2) + cos(lat1) cos(lat2) sin^2(dlon/2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;d = 2 R asin(sqrt(a)),   R = 6371 km  
&nbsp;&nbsp;&nbsp;- Scaling: Standardization z = (x - mean)/std, Normalization (min-max)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;x' = (x - min)/(max - min). Needed for distance/gradient based models  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(linear regression with GD, KNN, SVM, NN), not for tree models.  

### 3. OUTLIERS (Unit V - Outlier analysis)
&nbsp;&nbsp;&nbsp;An outlier is an observation far away from the rest of the data.  
&nbsp;&nbsp;&nbsp;Detection: box-plot / IQR rule  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;IQR = Q3 - Q1 ; outlier if x &lt; Q1 - 1.5 IQR or x &gt; Q3 + 1.5 IQR  
&nbsp;&nbsp;&nbsp;or Z-score |z| &gt; 3, or Isolation Forest / Local Outlier Factor.  
&nbsp;&nbsp;&nbsp;Treatment: remove, or CAP (clip) values to the IQR limits (done here).  

### 4. CORRELATION
&nbsp;&nbsp;&nbsp;Pearson correlation r = cov(X,Y) / (sigma\_X sigma\_Y), -1 &lt;= r &lt;= 1.  
&nbsp;&nbsp;&nbsp;r ~ +1 strong positive, r ~ -1 strong negative, r ~ 0 no linear relation.  
&nbsp;&nbsp;&nbsp;Visualised using a heat-map; helps feature selection and detecting  
&nbsp;&nbsp;&nbsp;multicollinearity.  

### 5. LINEAR REGRESSION (Unit III)
&nbsp;&nbsp;&nbsp;Models y = b0 + b1 x1 + ... + bn xn + e.  
&nbsp;&nbsp;&nbsp;Parameters chosen to minimise Mean Squared Error (Ordinary Least Squares):  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;J(b) = (1/m) SUM (y\_i - y\_hat\_i)^2  
&nbsp;&nbsp;&nbsp;Closed-form: b = (X^T X)^-1 X^T y, or iteratively by GRADIENT DESCENT.  
&nbsp;&nbsp;&nbsp;Regularised variants: RIDGE (L2 penalty lambda SUM b^2, shrinks weights),  
&nbsp;&nbsp;&nbsp;LASSO (L1 penalty lambda SUM |b|, can make weights exactly 0 -&gt; feature  
&nbsp;&nbsp;&nbsp;selection).  

### 6. RANDOM FOREST REGRESSION (Unit IV - Ensemble learning)
&nbsp;&nbsp;&nbsp;ENSEMBLE: combine many weak learners to get a strong learner.  
&nbsp;&nbsp;&nbsp;BAGGING (Bootstrap AGGregatING): train each model on a bootstrap sample  
&nbsp;&nbsp;&nbsp;(sampling with replacement) and average predictions -&gt; reduces variance.  
&nbsp;&nbsp;&nbsp;RANDOM FOREST = bagging of decision trees + random subset of features at  
&nbsp;&nbsp;&nbsp;each split. Prediction = average of all trees (regression).  
&nbsp;&nbsp;&nbsp;Handles non-linearity, robust to outliers, no scaling needed.  

### 7. BIAS, VARIANCE, OVER/UNDER-FITTING (Unit III)
&nbsp;&nbsp;&nbsp;Bias = error from wrong assumptions (underfitting: model too simple).  
&nbsp;&nbsp;&nbsp;Variance = error from sensitivity to training data (overfitting: model  
&nbsp;&nbsp;&nbsp;too complex, memorises noise). Goal: good GENERALIZATION (low test error);  
&nbsp;&nbsp;&nbsp;Total error = Bias^2 + Variance + irreducible noise.  

### 8. EVALUATION METRICS FOR REGRESSION
&nbsp;&nbsp;&nbsp;MAE  = (1/n) SUM |y - y\_hat|  
&nbsp;&nbsp;&nbsp;MSE  = (1/n) SUM (y - y\_hat)^2  
&nbsp;&nbsp;&nbsp;RMSE = sqrt(MSE)   (same units as target, penalises large errors)  
&nbsp;&nbsp;&nbsp;R^2  = 1 - SS\_res/SS\_tot   (fraction of variance explained, 1 = perfect)  

### Conclusion
After cleaning, outlier capping and distance feature extraction,
Random Forest generally gives higher R2 and lower RMSE than Linear  
Regression because fare depends non-linearly on distance/time.

In [ ]:
%matplotlib inline

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

DATA = "uber.csv"   # CSV file kept next to this notebook

### 1. Read dataset

In [ ]:
df = pd.read_csv(DATA)
print("Shape:", df.shape)
print(df.head())
df.info()

### 2. Pre-processing

In [ ]:
df = df.drop(columns=[c for c in ["Unnamed: 0", "key"] if c in df.columns])
print("\nMissing values:\n", df.isnull().sum())
df = df.dropna()

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
df = df.dropna(subset=["pickup_datetime"])
df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["weekday"] = df["pickup_datetime"].dt.dayofweek
df = df.drop(columns=["pickup_datetime"])

# remove impossible values (rides are in New York City: lat ~40-42,
# lon ~-75 to -72; some rows have swapped / zero coordinates)
df = df[(df["fare_amount"] > 0) &
        (df["passenger_count"] > 0) & (df["passenger_count"] <= 6) &
        df["pickup_latitude"].between(40, 42) &
        df["dropoff_latitude"].between(40, 42) &
        df["pickup_longitude"].between(-75, -72) &
        df["dropoff_longitude"].between(-75, -72)]


def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    a = (np.sin((lat2 - lat1) / 2) ** 2 +
         np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * 6371 * np.arcsin(np.sqrt(a))


df["distance_km"] = haversine(df["pickup_longitude"], df["pickup_latitude"],
                              df["dropoff_longitude"], df["dropoff_latitude"])
df = df[df["distance_km"] > 0]
print("\nShape after cleaning:", df.shape)
print(df.describe().T)

### 3. Identify outliers (box plots + IQR)

In [ ]:
num_cols = ["fare_amount", "distance_km", "pickup_longitude",
            "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
            "passenger_count"]
df[num_cols].plot(kind="box", subplots=True, layout=(2, 4), figsize=(16, 7),
                  title="Box plots BEFORE outlier treatment")
plt.tight_layout()
plt.show()


def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


print("\nOutlier count per column (IQR rule):")
for c in num_cols:
    lo, hi = iqr_bounds(df[c])
    print(f"  {c:<20} {((df[c] < lo) | (df[c] > hi)).sum()}")
    if c != "passenger_count":          # discrete 1-6, already validated
        df[c] = df[c].clip(lo, hi)      # cap outliers

df[num_cols].plot(kind="box", subplots=True, layout=(2, 4), figsize=(16, 7),
                  title="Box plots AFTER outlier treatment")
plt.tight_layout()
plt.show()

### 4. Correlation

In [ ]:
corr = df.corr()
print("\nCorrelation with fare_amount:\n",
      corr["fare_amount"].sort_values(ascending=False))
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heat-map")
plt.show()

### 5. Train-test split & scaling

In [ ]:
X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

### 6. Models

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)

rf = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1,
                           random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

### 7. Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"Model": name,
            "R2": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mse,
            "RMSE": np.sqrt(mse)}


results = pd.DataFrame([evaluate("Linear Regression", y_test, y_pred_lr),
                        evaluate("Random Forest", y_test, y_pred_rf)])
print("\nModel comparison:\n", results.to_string(index=False))

print("\nLinear regression coefficients:")
for f, c in zip(X.columns, lr.coef_):
    print(f"  {f:<20} {c:8.3f}")

print("\nRandom forest feature importance:")
for f, imp in sorted(zip(X.columns, rf.feature_importances_),
                     key=lambda t: -t[1]):
    print(f"  {f:<20} {imp:.3f}")

plt.figure(figsize=(12, 5))
for i, (name, pred) in enumerate([("Linear Regression", y_pred_lr),
                                  ("Random Forest", y_pred_rf)]):
    plt.subplot(1, 2, i + 1)
    plt.scatter(y_test, pred, s=3, alpha=0.3)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r")
    plt.xlabel("Actual fare")
    plt.ylabel("Predicted fare")
    plt.title(name)
plt.tight_layout()
plt.show()

best = results.loc[results["R2"].idxmax(), "Model"]
print(f"\nBetter model: {best}")